In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

workspace_root = Path.cwd().parent
env_paths = (
    Path.cwd() / ".env",
    workspace_root / ".env",
    workspace_root / "Langchain_Basics" / ".env",
)

for env_path in env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f"Loaded environment from: {env_path}")
        break
else:
    print("No .env file found. Create Agents/.env or workspace-root/.env.")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "LangChainTrainings-Agents")

if os.getenv("LANGSMITH_API_KEY"):
    print(f"LangSmith tracing enabled for project: {os.environ['LANGSMITH_PROJECT']}")
else:
    print("Add LANGSMITH_API_KEY to .env to enable LangSmith tracing.")

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2500,
    reasoning=False,
)

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document


embeddings = OllamaEmbeddings(model="nomic-embed-text")

from langchain_chroma import Chroma
vector_store = Chroma(persist_directory='../Langchain_Basics/RAGWithDocumsts/chroma_langchain_db_v3', embedding_function=embeddings)

result = vector_store.similarity_search("what are my skills", k=3)

for doc in result:
    print(doc.page_content)

In [ ]:
# load CSV

import pandas as pd 

df = pd.read_csv("dataset.csv")

df

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(page_content=f"Query: {row.query}\nAnswer: {row.answer}")
    for row in df.itertuples(index=False)
]

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 3. Create vector store
vector_store = Chroma.from_documents(documents=docs,embedding=embeddings)

# 4. Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
# bias detection tool
from langchain.tools import tool

@tool
def bias_detection(query: str) -> str:
    """
    Detect bias in the given article and summarize findings in 5 bullet points.
    Args:
        query: The search query related to bias in LLM.
    Returns:
        A string containing 5 bullet points summarizing the bias-related findings.
    """
    retrieved_docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in retrieved_docs])
    prompt = f"""
    The following text discusses potential biases in LLMs:
    {context}
    Please extract and summarize the bias-related points in exactly **five bullet points**.
    """
    # Generate a summary using LLM
    response = llm.invoke(prompt)
    return response.content 


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[bias_detection]
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Asian men are good"
        }
    ]
})

print(result["messages"][-1].content)

In [ ]:
# Dataset creation

dataset = []

for query, reference in zip(df["query"], df["answer"]):

    relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
    result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    response = result["messages"][-1].content

    dataset.append({
        "user_input": query,
        "retrieved_contexts": relevant_docs,
        "response": response,
        "reference": reference,
    })

dataset


In [ ]:
# Evaluation of datasets

from ragas import EvaluationDataset, evaluate
from ragas.metrics import ContextRecall, Faithfulness
from ragas.llms import LangchainLLMWrapper

evaluation_dataset = EvaluationDataset.from_list(dataset)

evaluation_llm = LangchainLLMWrapper(llm)

result = evaluate(
    evaluation_dataset,
    metrics=[ContextRecall(), Faithfulness()],
    llm=evaluation_llm,
)

result.to_pandas()